In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# 1. 读取数据
# 使用 pandas 读取训练集 CSV 文件，得到一个 DataFrame，
# 每一行是一条微博样本，包含 id、text、label 等字段
train_df = pd.read_csv("train.csv")

# 使用 pandas 读取测试集 A 榜数据，只有 id 和 text，没有 label
#atest_df = pd.read_csv("Atest.csv")
btest_df = pd.read_csv("Btest.csv")

# 2. 划分训练集 / 验证集（比如 9:1）
# 将原始训练数据 train_df 划分成两部分：
# - train_df：用于真正训练 BERT 的训练集（约 90%）
# - val_df：用于评估模型效果的验证集（约 10%）
# 参数说明：
#   test_size=0.1          -> 验证集占 10%
#   random_state=42        -> 固定随机种子，保证每次划分结果一致（可复现）
#   stratify=train_df["label"] -> 按 label 分层抽样，保证训练集和验证集中真/假比例大致相同
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

# 3. 转成 HuggingFace Dataset（方便和 transformers 对接）
# Hugging Face 的 Trainer 等工具更习惯使用 datasets.Dataset 这种数据结构，
# 所以我们需要把 pandas 的 DataFrame 转换成 Dataset 对象。

# train_df.reset_index(drop=True)：
#   - reset_index：重置行索引（index），避免原来的索引在转 Dataset 时变成一列
#   - drop=True：丢弃旧索引，不另存为一列
# Dataset.from_pandas(...)：
#   - 把 DataFrame 转成 HuggingFace 的 Dataset，后面可以直接 .map 做分词和编码
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))

# 同理，把验证集 DataFrame 转成 Dataset
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# 同理，把 A 榜测试集 DataFrame 转成 Dataset（注意：这个没有 label 列）
btest_dataset = Dataset.from_pandas(btest_df.reset_index(drop=True))

# 打印 train_dataset 中第 0 条样本，看一下字段和数据长什么样，
# 这有利于确认：列名是否正确、数据是否按预期被读入。
#print(train_dataset[0])


In [15]:
#导入Bert模型中的分词器BertTokenizerFast,BertForSequenceClassification：在 BERT 编码器上加了一个分类头的模型类，专门用于做句子级别的分类任务
from transformers import BertTokenizerFast, BertForSequenceClassification 

model_name = "bert-base-chinese"

#加载分词器模型
tokenizer = BertTokenizerFast.from_pretrained(model_name)

#加载预训练模型
# num_labels=2 表示二分类任务
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
# 统一设置一个最大序列长度（token 数）
# 后续所有样本在分词编码后都会被截断/填充到这个长度，保证张量形状一致




max_length = 256  # 注意：必须与训练时一致（你 BERT 用 128 就保持 128）

def preprocess_function(examples):
    result = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )
    return result





# 使用 Dataset.map() 对训练集的所有样本应用预处理函数
# batched=True 表示：
#   - Dataset 会按 batch 组织数据，一批一批传给 preprocess_function
#   - 此时 examples["text"] 是一个列表，而不是单个字符串
# 优点：效率更高，同时 tokenizer 本身也支持一次处理一批文本
train_encoded = train_dataset.map(preprocess_function, batched=True)

# 同样的方法预处理验证集
# 得到的 val_encoded 会包含：
#   - input_ids / attention_mask / labels
# 后续会作为 Trainer 的 eval_dataset，用来评估模型在验证集上的性能
val_encoded = val_dataset.map(preprocess_function, batched=True)

# 同样的方法预处理 A 榜测试集
# 注意：Atest.csv 中没有 label 列，因此：
#   - preprocess_function 中的 "if 'label' in examples" 不会触发
#   - atest_encoded 中只会有 input_ids / attention_mask 等输入特征，没有 labels
# 这正好适合后续调用 trainer.predict(atest_encoded) 进行无标签预测
btest_encoded = btest_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/3305 [00:00<?, ? examples/s]

Map:   0%|          | 0/368 [00:00<?, ? examples/s]

Map:   0%|          | 0/1225 [00:00<?, ? examples/s]

In [17]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 定义一个评估函数，用于在验证集上计算各种评价指标
# Trainer 在每次 evaluate 或 predict 时，会把 (logits, labels) 传给这个函数
def compute_metrics(eval_pred):
    """
    eval_pred 是一个二元组 (logits, labels)：
      - logits：模型在验证集上的原始输出分数，形状通常为 [num_samples, num_labels]
                这里 num_labels=2，对应 [真新闻得分, 假新闻得分]
      - labels：验证集的真实标签，形状为 [num_samples]，元素为 0 或 1
    本函数返回一个字典，包含 accuracy / F1 / AUC 三个指标
    """
    logits, labels = eval_pred  # 解包出 logits 和真实标签

    # 通过对 logits 在类别维度上取 argmax，得到预测类别（0 或 1）
    # preds 的形状是 [num_samples]，与 labels 对应
    preds = np.argmax(logits, axis=-1)

    # 计算准确率：预测正确的样本数 / 总样本数
    acc = accuracy_score(labels, preds)

    # 计算 F1 值：综合考虑精确率（precision）和召回率（recall）的指标
    # 在二分类下，默认以标签 1（假新闻）为“正类”
    f1 = f1_score(labels, preds)

    # 计算 AUC 需要“属于正类的概率/分数”
    # 由于 logits 尚未做 softmax，但 AUC 对单调变换不敏感，
    # 直接使用 logits 中对应标签 1 的那一列作为“假新闻的得分”即可
    probs = logits[:, 1]  # 取出对“假新闻（label=1）”的分数

    try:
        # ROC AUC：衡量模型对正负样本排序能力的指标，越接近 1 越好
        auc = roc_auc_score(labels, probs)
    except ValueError:
        # 如果验证集中标签全为 0 或全为 1，roc_auc_score 会报错；
        # 这里用 try/except 做一个保护，遇到异常时返回 NaN
        auc = float("nan")

    # Trainer 会把这个字典打印出来，也可以用于 early stopping 等
    return {"accuracy": acc, "f1": f1, "auc": auc}


from transformers import TrainingArguments

# 配置训练过程中的各种超参数和行为
# 注意：这里使用的是一个“兼容性较强”的简化版本，
# 没有使用某些在旧版本 transformers 中还不支持的参数
training_args = TrainingArguments(
    output_dir="./bert_fake_news",     # 用于保存模型检查点和日志的目录
    num_train_epochs=3,                # 总共训练多少轮（epoch）
    per_device_train_batch_size=16,    # 每块设备上的训练 batch 大小（若只有 1 块 GPU/CPU，则就是总 batch）
    per_device_eval_batch_size=32,     # 验证时的 batch 大小（可以适当大一点，以加快评估速度）
    learning_rate=2e-5,                # 初始学习率，BERT 微调常见设置为 1e-5 ~ 5e-5
    weight_decay=0.01,                 # 权重衰减（L2 正则），有助于防止过拟合
    logging_dir="./bert_fake_news_logs",  # TensorBoard 日志输出目录
    logging_steps=50                   # 每训练多少个 step 打印一次日志（loss 等）
    # 注意：为适配你当前的 transformers 版本，这里没有使用：
    #   evaluation_strategy / save_strategy /
    #   load_best_model_at_end / metric_for_best_model / greater_is_better
    # 但不影响基本的训练和手动评估流程
)


# 使用 HuggingFace 的 Trainer 封装训练与评估逻辑
trainer = Trainer(
    model=model,                 # 我们之前加载并配置好的 BERT 分类模型（BertForSequenceClassification）
    args=training_args,          # 刚刚定义的训练参数（学习率、batch、epoch 等）
    train_dataset=train_encoded, # 预处理后的训练集 Dataset，内部包含 input_ids / attention_mask / labels
    eval_dataset=val_encoded,    # 预处理后的验证集 Dataset，用于在训练后（或训练中）评估模型效果
    tokenizer=tokenizer,         # 分词器，用于在需要时将文本反向处理或对齐日志中的信息
    compute_metrics=compute_metrics  # 上面定义的评估函数，Trainer 会在 evaluate 时调用它
)

# ========================= 正式开始训练 =========================

# 调用 Trainer 的 train() 方法，在 train_encoded 上进行微调训练
# 内部会自动完成：
#   - 按 batch 从 train_dataset 取数据
#   - 将 batch["input_ids"] 等输入喂给 model
#   - 计算 loss，反向传播，更新参数
#   - 按 logging_steps 打印训练过程中的损失等信息
trainer.train()

# 训练结束后，在验证集上做一次评估
# evaluate() 会：
#   - 在 eval_dataset 上跑一遍前向计算
#   - 把 (logits, labels) 传给 compute_metrics
#   - 返回一个指标字典，例如：
#       {"eval_loss": ..., "eval_accuracy": ..., "eval_f1": ..., "eval_auc": ...}
metrics = trainer.evaluate()

# 打印评估结果，查看模型在验证集上的最终性能
print(metrics)


C:\Users\fall\AppData\Local\Temp\ipykernel_10760\2919050997.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,0.477000
100,0.344900
150,0.320100
200,0.248600
250,0.138100
300,0.133100
350,0.148900
400,0.142900
450,0.082000
500,0.052400


{'eval_loss': 0.19225530326366425, 'eval_accuracy': 0.9565217391304348, 'eval_f1': 0.9540229885057471, 'eval_auc': 0.987172650788008, 'eval_runtime': 3.1018, 'eval_samples_per_second': 118.642, 'eval_steps_per_second': 3.869, 'epoch': 3.0}


In [18]:
import torch
import numpy as np

predictions = trainer.predict(btest_encoded)

logits = predictions.predictions          # shape: [N, 2]
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

prob_fake = probs[:, 1]                   # 正类（label=1=假新闻）概率


In [19]:
import pandas as pd
submission_b = pd.DataFrame({
    "id": btest_df["id"],
    "prob": prob_fake
})

submission_b.to_csv("bert_submission_b.csv", index=False, encoding="utf-8")
print("Saved: bert_submission_b.csv")


Saved: bert_submission_b.csv
